## Fitting a Vol Surface with Heston

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange
from scipy.optimize import minimize
import pandas as pd
import QuantLib as ql
from py_vollib.black_scholes.implied_volatility import implied_volatility
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata

### Load and prepae the vol surface

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfopt.columns

In [ ]:
dfopt['quote_date'].unique()

In [ ]:
dfvol = dfopt[dfopt['quote_date'] == '2013-03-27']
dfvol = dfvol[dfvol['type'] == 'call']
underlying = dfvol['underlying'].unique()

In [ ]:
drop_list = ['contract', 'underlying', 'expiration', 'type', 'style',
       'bid', 'ask', 'volume', 'open_interest', 'quote_date', 'delta', 'gamma',
       'theta', 'vega', 'rate', '20D_HV','40D_HV', '60D_HV', 'BlackScholes', 'mid', 'error']
dfvol.drop(drop_list, axis=1, inplace=True)

In [ ]:
dfvol.head()

In [ ]:
dfvol['relative_strike'] = dfvol['strike'] / underlying.item()

In [ ]:
dfvol.drop(['strike'], axis=1, inplace=True)

In [ ]:
dfvol = dfvol[dfvol['relative_strike'] < 1.5]
dfvol = dfvol[dfvol['relative_strike'] > 0.5]
dfvol = dfvol[dfvol['maturity'] > 0.1]

In [ ]:
dfvol

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

xi = np.linspace(dfvol['maturity'].min(), dfvol['maturity'].max(), 100)
yi = np.linspace(dfvol['relative_strike'].min(), dfvol['relative_strike'].max(), 100)
xi, yi = np.meshgrid(xi, yi)
zi = griddata((dfvol['maturity'], dfvol['relative_strike']), dfvol['implied_volatility'], (xi, yi), method='cubic')

surf = ax.plot_surface(xi, yi, zi, cmap='viridis', edgecolor='none')
ax.set_xlabel('Maturity')
ax.set_ylabel('Relative Strike')
ax.set_zlabel('Implied Volatility')
ax.set_title('SPX Volatility Surface')

fig.colorbar(surf, shrink=0.5, aspect=5)

plt.show()

In [ ]:
def hestonIV(S0, r, q, tau, K, V0, lamda, vbar, eta, rho):
    todaysDate = ql.Date(17, 2, 2024)
    ql.Settings.instance().evaluationDate = todaysDate
    maturity = int(tau * 365)
    settlementDate = todaysDate 
    day_count = ql.Actual365Fixed()
    rfr = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, r, day_count))
    div_yield = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, q, day_count))
    
    exercise = ql.EuropeanExercise(todaysDate + maturity)
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    european_option = ql.VanillaOption(payoff, exercise)
        
    spot = ql.QuoteHandle(ql.SimpleQuote(S0))
    heston_process = ql.HestonProcess(rfr, div_yield, spot, V0, lamda, vbar, eta, rho)
    engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
    european_option.setPricingEngine(engine)
    price = european_option.NPV()
    iv = implied_volatility(price, S0, K, tau, r, 'c')
    return iv

In [ ]:
def objective_function(params, iv_surface):
    total_error = 0
    for i, row in iv_surface.iterrows():
        market_iv = row['implied_volatility']
        maturity = row['maturity']
        relative_strike = row['relative_strike']
        V0 = params[0]
        lamda = params[1]
        vbar = params[2]
        eta = params[3]
        rho = params[4]
        
        try:
            model_iv = hestonIV(1.0, 0.0, 0.0, maturity, relative_strike, 
                               V0, lamda, vbar, eta, rho)
            error = (market_iv - model_iv) ** 2
        except Exception as e:
            print("Error for params: {}, mat: {}, strike: {}".format(params, maturity, relative_strike))
            error = np.inf  # Assign a high error for failures
        total_error += error
    return total_error

In [ ]:
eta = 2.5
rho = -0.5
lamda = 5.0
vbar = 0.5
V0 = 0.5
param_lst = [V0, lamda, vbar, eta, rho]
initial_params = np.array(param_lst)

In [ ]:
S0 = 1.0
r = 0.0
q = 0.0
tau = 0.14236824
K = 0.5

In [ ]:
hestonIV(S0, r, q, tau, K, V0, lamda, vbar, eta, rho)

In [ ]:
optimization_method = 'Nelder-Mead'

In [ ]:
bounds = [[0.0, 1.0], [0.0, 10.0], [0.0, 1.0], [0.0, 5.0], [-1.0, 0.0]]

In [ ]:
result = minimize(objective_function, initial_params, args=(dfvol,), bounds=bounds, method=optimization_method)

In [ ]:
if result.success:
    fitted_params = result.x
    print("Optimization succeeded.")
    print(f"Fitted Heston parameters: {fitted_params}")
else:
    print("Optimization failed.")
    print(result.message)

In [ ]:
V0f, lamdaf, vbarf, etaf, rhof = result.x

In [ ]:
df_hest = pd.DataFrame()
for x_val, y_val in np.nditer([xi, yi]):
    try:
        iv = hestonIV(S0, r, q, x_val.item(), y_val.item(), V0f, lamdaf, vbarf, etaf, rhof)
        new_row = {'implied_volatility': iv, 'maturity': x_val, 'relative_strike': y_val}
        df_hest.append(new_row, ignore_index=True)
    except:
        continue

In [ ]:
df_hest